The spelled-out intro to neural networks and backpropagation - Part 4 [link](https://youtu.be/VMj-3S1tku0?t=6235)

In [11]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [12]:
# @title visulizing the expression
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right

  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot

In [13]:
# @title Value class
class Value:

  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"

  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')

    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward

    return out

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')

    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward

    return out

  def __pow__(self, other):
    assert isinstance(other, (int, float)), "only supporting int/float powers for now"
    out = Value(self.data**other, (self,), f'**{other}')

    def _backward():
        self.grad += other * (self.data ** (other - 1)) * out.grad
    out._backward = _backward

    return out

  def __rmul__(self, other): # other * self
    return self * other

  def __truediv__(self, other): # self / other
    return self * other**-1

  def __neg__(self): # -self
    return self * -1

  def __sub__(self, other): # self - other
    return self + (-other)

  def __radd__(self, other): # other + self
    return self + other

  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self, ), 'tanh')

    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward

    return out

  def exp(self):
    x = self.data
    out = Value(math.exp(x), (self, ), 'exp')

    def _backward():
      self.grad += out.data * out.grad # NOTE: in the video I incorrectly used = instead of +=. Fixed here.
    out._backward = _backward

    return out


  def backward(self):

    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)

    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

Part 4
---

In [14]:
import torch
import random

In [15]:
# custom neural network classes

class Neuron: # a neuron that can have multiple inputs, weights and a bias

  # param 'nin' is the no of input to a neuron
  def __init__(self, nin):
    # randomely creating weights value and a bias value
    self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
    self.b = Value(random.uniform(-1,1))

  # __call__ make it possible to call nueron object to be called like a function
  def __call__(self, x):
    # summazation w * x + b
    act = sum((wi*xi for wi, xi in zip(self.w, x)))+ self.b
    # activation function
    out = act.tanh()
    return out

  # provide parameters(list containing weights and bias) in a neuron
  def parameters(self):
    return self.w + [self.b] # return a list of weights and bias

class Layer: # layer that contains neurons

  # param 'nin' is the no of input to a neuron and 'nout' the no of neurons in this layer
  def __init__(self, nin, nout):
    self.neurons = [Neuron(nin) for _ in range(nout)]

  def __call__(self, x):
    outs = [n(x) for n in self.neurons] # individuly evulating each neuron
    return outs[0] if len(outs) == 1 else outs

  # iterate over neurons in layer and then iterate over params in each neurons
  def parameters(self):
    return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP: # Mult layer perceptron , that has layers

  # param 'nin' is the no of input to a neuron and 'nouts' is a list of no of neurons in layer
  def __init__(self, nin, nouts):
    sz = [nin] + nouts
    self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

  def __call__(self, x):
    for layer in self.layers: # calling layers sequentially
      x = layer(x)
    return x

  # iterate over layers in MLP and then iterate over params in each layer
  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()]


In [16]:
# so if we need a MLP with 3 neuron input layer, 2 hidden layer with 4 neurons each and single neuron output layer
x = [2.0, 3.0, -1.0] # inputs
n = MLP(3,[4,4,1])
n(x)


Value(data=0.9417176840407402)

In [17]:
#draw_dot(n(x))
# n.parameters() # will list all the parameters available for this neural network
len(n.parameters()) # total no of parameters

41

In [18]:
# four inputs
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]

# desired targets/ground truth
ys = [1.0, -1.0, -1.0, 1.0]

In [19]:
ypred = [n(x) for x in xs]
ypred

[Value(data=0.9417176840407402),
 Value(data=0.9588353458301144),
 Value(data=0.5069275098018061),
 Value(data=0.9490214549255678)]

----
Find loss

In [20]:
# individual losses . what we need is low losses
[(yout - ygt)**2 for ygt, yout in zip(ys, ypred)]

[Value(data=0.003396828353574991),
 Value(data=3.837035912073384),
 Value(data=2.2708305197974727),
 Value(data=0.002598812057905918)]

In [21]:
# loss
loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
loss

Value(data=6.113862072282338)

In [22]:
# note: The loss function is the starting point for backpropagation for neural network
# loss.backward()

In [23]:
# an individual weight value
n.layers[0].neurons[0].w[0].data
# an individual weight's grad
n.layers[0].neurons[0].w[0].grad


0.0

----
Minimizing loss/training the neural network

In [24]:
# predition before training
ypred

[Value(data=0.9417176840407402),
 Value(data=0.9588353458301144),
 Value(data=0.5069275098018061),
 Value(data=0.9490214549255678)]

In [25]:
# here we are
for k in range(20):

  # forward pass
  ypred = [n(x) for x in xs]
  loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))

  # backward pass
  for p in n.parameters():
    p.grad = 0.0 # we need do zero grad to rest the grads
  loss.backward()

  # update
  for p in n.parameters():
    p.data += -0.1 * p.grad # here we are nuding the weight by a small

  print(k, loss.data)

0 6.113862072282338
1 3.702531083881619
2 2.980884306802617
3 2.7011096958184724
4 3.3007087764483756
5 2.1754909909146765
6 2.5713760031096435
7 1.2295073909474281
8 0.09047900331461736
9 0.04330117808302011
10 0.030271705334119554
11 0.023517571430051346
12 0.019312363675072042
13 0.01642281051216005
14 0.014307933409005949
15 0.012689918792371933
16 0.011410517171632637
17 0.010372637150903335
18 0.009513267560238667
19 0.00878966862479397


In [26]:
# predition before training
ypred

[Value(data=0.9905985293163124),
 Value(data=-0.9276916569590878),
 Value(data=-0.9426878222643259),
 Value(data=0.9862850890032051)]

In [27]:
n.parameters()

[Value(data=-0.12348588524603407),
 Value(data=-0.6215538317061222),
 Value(data=1.2293927462514112),
 Value(data=0.4872352150164787),
 Value(data=0.03375945516862773),
 Value(data=0.4286386043671171),
 Value(data=-1.0866841179850177),
 Value(data=-0.14730549881516705),
 Value(data=-0.11014918615460957),
 Value(data=-0.029722160690685274),
 Value(data=1.0433013961486122),
 Value(data=0.013872317041757797),
 Value(data=0.609311817827027),
 Value(data=0.5685279183401888),
 Value(data=0.8275726119053162),
 Value(data=0.8313609632586525),
 Value(data=-0.07990080076775417),
 Value(data=-0.5868352518120508),
 Value(data=0.8438366900411777),
 Value(data=-0.5897324393611668),
 Value(data=-0.18821221084759565),
 Value(data=-0.047207216082692516),
 Value(data=1.0459735362199452),
 Value(data=-0.8984445273611555),
 Value(data=-0.24193742092529366),
 Value(data=-0.37760698063669684),
 Value(data=-1.0780325581299048),
 Value(data=0.1882860132653529),
 Value(data=-0.020994911297174273),
 Value(data=